In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

project_root = Path.cwd()
if project_root.name == "detecting_pipeline":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from detecting_pipeline.detector import Detector
from embedding_pipeline.embedder import Embedder

AUDIO_EXTS = {".wav", ".flac", ".mp3", ".m4a", ".aac", ".ogg", ".wma"}

In [2]:
def evaluate_detector_accuracy(detector, audio_files: list, expected_label: int, model_name: str = "model") -> None:
    total = len(audio_files)
    correct = sum(detector.detect(p)["pred_label"] == expected_label for p in audio_files)
    print(f"{model_name}: accuracy={correct / max(1, total):.4f} ({correct}/{total})")

In [3]:
max_files_per_folder = 50
support_files_per_folder = 10

# clustering/filter thresholds
similarity_threshold = 0.99 # cosine_similarity_threshold
min_cluster_size = 5 # minimum number of vectors to form a cluster
fake_ratio_threshold = 0.8 # minimum ratio of fake-labeled vectors in a cluster to consider it "mostly-fake"
top_k = 100
similar_ratio_threshold = 0.5

support_set = None


# load baseline vectors and metadata
data = torch.load("../baseline_vectors/detected_vectors_with_metadata_big.pt", map_location="cpu")
    
entries = data.get("metadata", [])
vectors = data.get("vectors")
vectors = vectors.float().view(vectors.size(0), -1)

entries, vectors = [dict(e) for e in entries], vectors.clone()


# initialize embedder and detector
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_dim = 128
embedder = Embedder(embedding_dim=embedding_dim,).to(device)
embedder.load_state_dict(torch.load("../embedding_pipeline/embedder_checkpoint.pth", map_location=device)["model"])

detector = Detector(
    raw_vectors=vectors.to(device), 
    labels=[entry["label"] for entry in entries],
    embedder=embedder,
    cosine_similarity_threshold=similarity_threshold,
    similar_ratio_threshold=similar_ratio_threshold,
    device=device
)

In [4]:
support_root = Path("../data/test_data/FlashSpeech")
support_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:support_files_per_folder]
for audio_path in tqdm(support_files, desc="update_memory FlashSpeech"):
    detector.update_memory(audio_path)

eval_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 1, "FlashSpeech")

update_memory FlashSpeech:   0%|          | 0/10 [00:00<?, ?it/s]c:\Users\whyam\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
update_memory FlashSpeech: 100%|██████████| 10/10 [11:41<00:00, 70.10s/it]
c:\Users\whyam\OneDrive\Desktop\pogramming\vs projects\deepfake-audio-detection\detecting_pipeline\detector.py:88: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ref = torch.tensor(ref).float().to(query.device)


16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128])
16541 reference vectors in memory
torch.Size([16541, 128

In [5]:
support_root = Path("../data/test_data/NaturalSpeech3")
support_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:support_files_per_folder]
for audio_path in tqdm(support_files, desc="update_memory NaturalSpeech3"):
    detector.update_memory(audio_path)

eval_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 1, "NaturalSpeech3")

update_memory NaturalSpeech3:   0%|          | 0/10 [00:06<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
root = Path("../data/test_data/real_samples")

eval_files = sorted(
    p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 0, "real_samples")

In [ ]:
support_root = Path("../data/test_data/OpenAI")
support_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:support_files_per_folder]
for audio_path in tqdm(support_files, desc="update_memory OpenAI"):
    detector.update_memory(audio_path)

eval_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 1, "OpenAI")

[HNSW] Building index for 100360 vectors...
[HNSW] Querying top-100 neighbors per vector...


[HNSW] Building graph: 100%|██████████| 100360/100360 [00:11<00:00, 9009.92it/s] 


[HNSW] Generating clusters...
[HNSW] clusters=25 remaining=100183
[HNSW] clusters=25 remaining=100182
[HNSW] clusters=25 remaining=100181
[HNSW] clusters=50 remaining=99835
[HNSW] clusters=75 remaining=99309
[HNSW] clusters=75 remaining=99308
[HNSW] clusters=75 remaining=99307
[HNSW] clusters=75 remaining=99306
[HNSW] clusters=75 remaining=99305
[HNSW] clusters=100 remaining=98695
[HNSW] clusters=100 remaining=98694
[HNSW] clusters=125 remaining=98140
[HNSW] clusters=150 remaining=97510
[HNSW] clusters=150 remaining=97509
[HNSW] clusters=175 remaining=96992
[HNSW] clusters=175 remaining=96991
[HNSW] clusters=200 remaining=96488
[HNSW] clusters=225 remaining=95883
[HNSW] clusters=250 remaining=95208
[HNSW] clusters=250 remaining=95207
[HNSW] clusters=250 remaining=95206
[HNSW] clusters=275 remaining=94644
[HNSW] clusters=275 remaining=94643
[HNSW] clusters=300 remaining=94025
[HNSW] clusters=325 remaining=93424
[HNSW] clusters=350 remaining=92790
[HNSW] clusters=375 remaining=92197
[HNS

In [ ]:
support_root = Path("../data/test_data/PromptTTS2")
support_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:support_files_per_folder]
for audio_path in tqdm(support_files, desc="update_memory PromptTTS2"):
    detector.update_memory(audio_path)

eval_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 1, "PromptTTS2")

[HNSW] Building index for 100474 vectors...
[HNSW] Querying top-100 neighbors per vector...


[HNSW] Building graph: 100%|██████████| 100474/100474 [00:09<00:00, 10896.58it/s]


[HNSW] Generating clusters...
[HNSW] clusters=25 remaining=100297
[HNSW] clusters=50 remaining=99790
[HNSW] clusters=50 remaining=99789
[HNSW] clusters=50 remaining=99788
[HNSW] clusters=50 remaining=99787
[HNSW] clusters=50 remaining=99786
[HNSW] clusters=50 remaining=99785
[HNSW] clusters=50 remaining=99784
[HNSW] clusters=50 remaining=99783
[HNSW] clusters=50 remaining=99782
[HNSW] clusters=50 remaining=99781
[HNSW] clusters=50 remaining=99780
[HNSW] clusters=50 remaining=99779
[HNSW] clusters=50 remaining=99778
[HNSW] clusters=50 remaining=99777
[HNSW] clusters=50 remaining=99776
[HNSW] clusters=50 remaining=99775
[HNSW] clusters=50 remaining=99774
[HNSW] clusters=75 remaining=99203
[HNSW] clusters=75 remaining=99202
[HNSW] clusters=75 remaining=99201
[HNSW] clusters=100 remaining=98665
[HNSW] clusters=100 remaining=98664
[HNSW] clusters=125 remaining=98099
[HNSW] clusters=150 remaining=97462
[HNSW] clusters=175 remaining=96954
[HNSW] clusters=200 remaining=96443
[HNSW] clusters=20

In [ ]:
support_root = Path("../data/test_data/seedtts_files")
support_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:support_files_per_folder]
for audio_path in tqdm(support_files, desc="update_memory seedtts_files"):
    detector.update_memory(audio_path)

eval_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 1, "seedtts_files")

[HNSW] Building index for 100488 vectors...
[HNSW] Querying top-100 neighbors per vector...


[HNSW] Building graph: 100%|██████████| 100488/100488 [00:10<00:00, 9198.98it/s]


[HNSW] Generating clusters...
[HNSW] clusters=25 remaining=100312
[HNSW] clusters=50 remaining=99822
[HNSW] clusters=50 remaining=99821
[HNSW] clusters=50 remaining=99820
[HNSW] clusters=50 remaining=99819
[HNSW] clusters=75 remaining=99225
[HNSW] clusters=75 remaining=99224
[HNSW] clusters=75 remaining=99223
[HNSW] clusters=75 remaining=99222
[HNSW] clusters=100 remaining=98714
[HNSW] clusters=100 remaining=98713
[HNSW] clusters=125 remaining=98155
[HNSW] clusters=150 remaining=97721
[HNSW] clusters=175 remaining=97175
[HNSW] clusters=200 remaining=96535
[HNSW] clusters=225 remaining=95919
[HNSW] clusters=225 remaining=95918
[HNSW] clusters=250 remaining=95305
[HNSW] clusters=275 remaining=94719
[HNSW] clusters=300 remaining=94197
[HNSW] clusters=300 remaining=94196
[HNSW] clusters=325 remaining=93594
[HNSW] clusters=350 remaining=92945
[HNSW] clusters=375 remaining=92353
[HNSW] clusters=400 remaining=91822
[HNSW] clusters=425 remaining=91173
[HNSW] clusters=450 remaining=90561
[HNSW]

In [ ]:
support_root = Path("../data/test_data/VALLE")
support_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:support_files_per_folder]
for audio_path in tqdm(support_files, desc="update_memory VALLE"):
    detector.update_memory(audio_path)

eval_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 1, "VALLE")

[HNSW] Building index for 100533 vectors...
[HNSW] Querying top-100 neighbors per vector...


[HNSW] Building graph: 100%|██████████| 100533/100533 [00:09<00:00, 11090.63it/s]


[HNSW] Generating clusters...
[HNSW] clusters=25 remaining=100360
[HNSW] clusters=50 remaining=99998
[HNSW] clusters=50 remaining=99997
[HNSW] clusters=50 remaining=99996
[HNSW] clusters=75 remaining=99432
[HNSW] clusters=75 remaining=99431
[HNSW] clusters=100 remaining=98855
[HNSW] clusters=100 remaining=98854
[HNSW] clusters=100 remaining=98853
[HNSW] clusters=100 remaining=98852
[HNSW] clusters=100 remaining=98851
[HNSW] clusters=125 remaining=98409
[HNSW] clusters=150 remaining=97937
[HNSW] clusters=175 remaining=97364
[HNSW] clusters=200 remaining=96770
[HNSW] clusters=225 remaining=96115
[HNSW] clusters=250 remaining=95514
[HNSW] clusters=250 remaining=95513
[HNSW] clusters=275 remaining=94922
[HNSW] clusters=300 remaining=94345
[HNSW] clusters=325 remaining=93708
[HNSW] clusters=350 remaining=93139
[HNSW] clusters=375 remaining=92514
[HNSW] clusters=400 remaining=91855
[HNSW] clusters=425 remaining=91241
[HNSW] clusters=450 remaining=90708
[HNSW] clusters=475 remaining=90218
[HN

In [ ]:
support_root = Path("../data/test_data/VoiceBox")
support_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:support_files_per_folder]
for audio_path in tqdm(support_files, desc="update_memory VoiceBox"):
    detector.update_memory(audio_path)

eval_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 1, "VoiceBox")

[HNSW] Building index for 100552 vectors...
[HNSW] Querying top-100 neighbors per vector...


[HNSW] Building graph: 100%|██████████| 100552/100552 [00:14<00:00, 6996.09it/s]


[HNSW] Generating clusters...
[HNSW] clusters=25 remaining=100376
[HNSW] clusters=50 remaining=100041
[HNSW] clusters=75 remaining=99553
[HNSW] clusters=75 remaining=99552
[HNSW] clusters=100 remaining=98969
[HNSW] clusters=125 remaining=98400
[HNSW] clusters=150 remaining=97839
[HNSW] clusters=175 remaining=97279
[HNSW] clusters=200 remaining=96725
[HNSW] clusters=225 remaining=96110
[HNSW] clusters=250 remaining=95525
[HNSW] clusters=275 remaining=94941
[HNSW] clusters=275 remaining=94940
[HNSW] clusters=275 remaining=94939
[HNSW] clusters=275 remaining=94938
[HNSW] clusters=300 remaining=94358
[HNSW] clusters=325 remaining=93780
[HNSW] clusters=350 remaining=93125
[HNSW] clusters=375 remaining=92532
[HNSW] clusters=400 remaining=91905
[HNSW] clusters=425 remaining=91309
[HNSW] clusters=450 remaining=90674
[HNSW] clusters=475 remaining=90056
[HNSW] clusters=500 remaining=89432
[HNSW] clusters=525 remaining=88849
[HNSW] clusters=550 remaining=88216
[HNSW] clusters=575 remaining=87641


In [ ]:
support_root = Path("../data/test_data/xTTS")
support_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:support_files_per_folder]
for audio_path in tqdm(support_files, desc="update_memory xTTS"):
    detector.update_memory(audio_path)

eval_files = sorted(
    p for p in support_root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS
)[:max_files_per_folder]
evaluate_detector_accuracy(detector, eval_files, 1, "xTTS")

[HNSW] Building index for 100559 vectors...
[HNSW] Querying top-100 neighbors per vector...


[HNSW] Building graph: 100%|██████████| 100559/100559 [00:11<00:00, 8494.55it/s]


[HNSW] Generating clusters...
[HNSW] clusters=25 remaining=100378
[HNSW] clusters=25 remaining=100377
[HNSW] clusters=50 remaining=100019
[HNSW] clusters=50 remaining=100018
[HNSW] clusters=50 remaining=100017
[HNSW] clusters=75 remaining=99418
[HNSW] clusters=100 remaining=98893
[HNSW] clusters=125 remaining=98365
[HNSW] clusters=125 remaining=98364
[HNSW] clusters=125 remaining=98363
[HNSW] clusters=125 remaining=98362
[HNSW] clusters=150 remaining=97862
[HNSW] clusters=175 remaining=97267
[HNSW] clusters=175 remaining=97266
[HNSW] clusters=200 remaining=96723
[HNSW] clusters=200 remaining=96722
[HNSW] clusters=200 remaining=96721
[HNSW] clusters=225 remaining=96134
[HNSW] clusters=250 remaining=95536
[HNSW] clusters=275 remaining=94964
[HNSW] clusters=300 remaining=94395
[HNSW] clusters=325 remaining=93763
[HNSW] clusters=350 remaining=93176
[HNSW] clusters=375 remaining=92587
[HNSW] clusters=400 remaining=91943
[HNSW] clusters=425 remaining=91345
[HNSW] clusters=450 remaining=90779